In [ ]:
# load the model definitions

%run 04_models_ABC.ipynb


In [ ]:
# load the cleaned data

scada = pd.read_parquet(
    SCADA_FILE
).copy()

static = pd.read_parquet(
    STATIC_FILE
).copy()

scada["timestamp"] = pd.to_datetime(
    scada["timestamp"],
    utc=True,
)

farm_data, turbine_ids = build_modelling_data(
    scada
)

print(f"farm timestamps: {len(farm_data):,}")
print(f"turbines: {turbine_ids}")


In [ ]:
# check the common chronological split

train_data, validation_data, test_data = chronological_split(
    farm_data
)

print(f"training rows: {len(train_data):,}")
print(f"validation rows: {len(validation_data):,}")
print(f"test rows: {len(test_data):,}")

print(
    f"training period: "
    f"{train_data.index.min()} to {train_data.index.max()}"
)

print(
    f"validation period: "
    f"{validation_data.index.min()} to {validation_data.index.max()}"
)

print(
    f"test period: "
    f"{test_data.index.min()} to {test_data.index.max()}"
)


In [ ]:
# run the final five-seed experiment

all_results = {}

for seed in SEEDS:
    print(f"running seed {seed}")

    all_results[seed] = run_models_ABC(
        farm_data=farm_data,
        scada=scada,
        static=static,
        turbine_ids=turbine_ids,
        seed=seed,
    )


In [ ]:
# save seed-level metrics

summary_rows = []

for seed, result in all_results.items():
    row = {
        "seed": seed,
    }

    for model_name in [
        "Model A",
        "Model B",
        "Model C",
    ]:
        model_key = model_name.replace(
            "Model ",
            "",
        )

        model_metrics = result[
            "metrics"
        ][model_name]

        row[
            f"{model_key}_MAE_kW"
        ] = model_metrics["MAE_kW"]

        row[
            f"{model_key}_RMSE_kW"
        ] = model_metrics["RMSE_kW"]

        row[
            f"{model_key}_R2"
        ] = model_metrics["R2"]

        row[
            f"{model_key}_bias_kW"
        ] = model_metrics["bias_kW"]

    row["B_ws_MAE_ms"] = result[
        "B_ws_metrics"
    ]["MAE_ms"]

    row["B_ws_R2"] = result[
        "B_ws_metrics"
    ]["R2"]

    row["A_best_epoch"] = result[
        "best_epochs"
    ]["Model A"]

    row["B_best_epoch"] = result[
        "best_epochs"
    ]["Model B"]

    row["C_best_epoch"] = result[
        "best_epochs"
    ]["Model C"]

    summary_rows.append(row)

summary = pd.DataFrame(
    summary_rows
).sort_values("seed")

summary.to_csv(
    RESULTS_DIR / "seed_summary.csv",
    index=False,
)

print(summary.round(4).to_string(index=False))


In [ ]:
# save the fixed baselines and oracle metrics

first_seed = SEEDS[0]

reference_metrics = pd.DataFrame(
    [
        {
            "reference": "Linear regression",
            **all_results[
                first_seed
            ]["metrics"]["Linear regression"],
        },
        {
            "reference": "Measured-speed reference",
            **all_results[
                first_seed
            ]["metrics"]["Measured-speed reference"],
        },
    ]
)

reference_metrics.to_csv(
    RESULTS_DIR / "reference_metrics.csv",
    index=False,
)

print(
    reference_metrics
    .round(4)
    .to_string(index=False)
)


In [ ]:
# save predictions required by the results and wake analyses

for seed, result in all_results.items():
    prediction_data = result[
        "test_data"
    ].copy()

    prediction_data["actual_farm_power"] = (
        result["actual_farm_power"]
    )

    prediction_data["pred_A_power"] = (
        result["pred_A_power"]
    )

    prediction_data["pred_B_power"] = (
        result["pred_B_power"]
    )

    prediction_data["pred_C_power"] = (
        result["pred_C_power"]
    )

    prediction_data["oracle_power"] = (
        result["oracle_power"]
    )

    for turbine_index, turbine_id in enumerate(
        turbine_ids
    ):
        prediction_data[
            f"actual_ws_t{turbine_id}"
        ] = result[
            "actual_B_ws"
        ][:, turbine_index]

        prediction_data[
            f"pred_B_ws_t{turbine_id}"
        ] = result[
            "pred_B_ws"
        ][:, turbine_index]

        prediction_data[
            f"actual_power_t{turbine_id}"
        ] = result[
            "actual_C_turbine_power"
        ][:, turbine_index]

        prediction_data[
            f"pred_B_power_t{turbine_id}"
        ] = result[
            "pred_B_turbine_power"
        ][:, turbine_index]

        prediction_data[
            f"pred_C_power_t{turbine_id}"
        ] = result[
            "pred_C_turbine_power"
        ][:, turbine_index]

    prediction_data.to_parquet(
        RESULTS_DIR
        / f"predictions_seed_{seed}.parquet"
    )


In [ ]:
# save training and validation histories

for seed, result in all_results.items():
    history = pd.DataFrame(
        {
            "A_train": pd.Series(
                result["loss_history"]["Model A"]["train"]
            ),
            "A_validation": pd.Series(
                result["loss_history"]["Model A"]["validation"]
            ),
            "B_train": pd.Series(
                result["loss_history"]["Model B"]["train"]
            ),
            "B_validation": pd.Series(
                result["loss_history"]["Model B"]["validation"]
            ),
            "C_train": pd.Series(
                result["loss_history"]["Model C"]["train"]
            ),
            "C_validation": pd.Series(
                result["loss_history"]["Model C"]["validation"]
            ),
        }
    )

    history.index.name = "epoch"

    history.to_csv(
        RESULTS_DIR
        / f"loss_history_seed_{seed}.csv"
    )


In [ ]:
# summarise performance across the five seeds

metric_columns = [
    column
    for column in summary.columns
    if column != "seed"
]

summary_stats = pd.DataFrame(
    {
        "mean": summary[
            metric_columns
        ].mean(),
        "std": summary[
            metric_columns
        ].std(ddof=1),
    }
)

summary_stats.to_csv(
    RESULTS_DIR / "mean_std_summary.csv"
)

print(
    summary_stats
    .round(4)
    .to_string()
)
